In [1]:
import pandas as pd
import numpy as np
from joblib import parallel_backend
import json
import os
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import datetime  # Added for timestamps
import time
warnings.filterwarnings('ignore')
df=pd.read_csv('df_eng.csv')

In [6]:
print("\nExecuting Regime & Outlier Analysis...")
DIRS = [
    "data_processed/split_definitions", 
    "results/models", "results/metrics", "results/predictions", 
    "results/tables", "manuscript_assets", "supplementary_assets"
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

    
DATA_FILE = "clean_data.csv"
ID_COL = "filename"
TARGET_COL = "uptake(mmol/g) methane at 65 bar"
TOPO_COL = "Crystalnet"
GEOMETRY_COLS = ['Density', 'AVAf', 'AVA', 'ASA', 'Df', 'Di']

preds = pd.read_csv("results/predictions/Table5_random_preds.csv")
merged = preds.merge(df, on=ID_COL, how="left")
merged['abs_error'] = (merged['y_true'] - merged['y_pred']).abs()


df['pld_decile'] = pd.qcut(df['Df'], 10, labels=False, duplicates='drop')
df['density_decile'] = pd.qcut(df['Density'], 10, labels=False, duplicates='drop')
df['sa_decile'] = pd.qcut(df['ASA'], 10, labels=False, duplicates='drop')
df['uptake_decile'] = pd.qcut(df[TARGET_COL], 10, labels=False, duplicates='drop')

regime_df = merged.copy()

# 2. Topology Frequency Check (Rare vs Common)
topo_freq_perf = regime_df.groupby(['family', 'model', 'topology_frequency_group'])['abs_error'].mean().reset_index()
topo_freq_perf.to_csv("results/tables/Topology_Frequency_Check.csv", index=False)

# 3. Failure/Outlier Anatomy (Top 1% error)
best_model_data = regime_df[regime_df['model'] == 'hgb']
geom_hgb = best_model_data[best_model_data['family'] == 'geometry_only']

avg_preds = geom_hgb.groupby(ID_COL)[['y_true', 'y_pred', 'abs_error']].mean().reset_index()
threshold = avg_preds['abs_error'].quantile(0.99) # Top 1% per methodology
outlier_ids = avg_preds[avg_preds['abs_error'] >= threshold][ID_COL]

df['is_outlier'] = df[ID_COL].isin(outlier_ids)
outlier_stats = df.groupby('is_outlier')[GEOMETRY_COLS + ['lcd_pld_ratio', 'cavity_window_gap']].agg(['mean', 'median']).T
outlier_stats.to_csv("results/tables/outlier_statistical_comparison.csv")

# --- MAIN FIGURES ---
raw_mets = pd.read_csv("results/metrics/Table5_random_raw.csv")


m=2.5

params = {
    # Font family
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],

    # Font sizes
    "axes.labelsize": 10*m,
    "font.size": 10*m,
    "legend.fontsize": 10*m,
    "xtick.labelsize": 9*m,
    "ytick.labelsize": 9*m,

    # Style for axis labels (xlabel, ylabel)
    'axes.labelweight': 'bold',
    'axes.labelcolor': 'black',

    # General styles for other elements
    'font.weight': 'bold',       # Makes title, etc., bold
    'xtick.color': 'black',      # Sets tick label color
    'ytick.color': 'black',
    'legend.labelcolor': 'black'
}

plt.rcParams.update(params)


plt.figure(figsize=(10, 6))
ax = sns.barplot(data=raw_mets, x='family', y='r2', hue='model', errorbar='sd')
plt.tight_layout(rect=[0, 0, 0.85, 1])  

ax.set_xticklabels(['Geo','Enriched','Topo','Geo + Topo'])
plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left') 
ax.set_ylabel('$R^2$',fontsize=11*3.5)
ax.set_xlabel('Descriptor Family',fontsize=11*3.5)

# قرار دادن لگند خارج از نمودار
plt.savefig("manuscript_assets/figure1_benchmark.pdf")
plt.close()


pivot_pld = best_model_data.pivot_table(index='family', columns='pld_decile', values='abs_error', aggfunc='mean')
plt.figure(figsize=(10*1.5,4*1.5))
ax=sns.heatmap(pivot_pld, annot=True, cmap="YlOrRd",    linewidths=0.7,      # اضافه کردن خطوط بین سلول‌ها
    linecolor='k',cbar_kws={'aspect': 30, 'shrink': 0.9, 'label': 'Absolute Error (mmol/g)' }) # CRITICAL: Added label})

ax.set_yticklabels(['Enriched','Geo','Geo + Topo','Topo'])
ax.set_xlabel('PLD Decile',fontsize=11*3.5)
ax.set_ylabel('Descriptor Family',fontsize=11*3.5)
plt.tight_layout()
plt.savefig("manuscript_assets/figure2_sufficiency_heatmap.pdf")
plt.close()




plt.figure(figsize=(10,6))
ax=sns.violinplot(data=geom_hgb, x='density_decile', y='abs_error')
ax.set_xlabel('Density Decile',fontsize=11*3.5)
ax.set_ylabel('Absolute Error',fontsize=11*3.5)
plt.tight_layout()
plt.savefig("manuscript_assets/figure3_error_anatomy.pdf")
plt.close()



plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df[df['is_outlier'] == False],
    x='Df',
    y='Di',
    alpha=0.3,  # Increased from 0.1 for visibility
    s=50,
    color='gray',
    zorder=0,
    edgecolor='none',
    label='Non-outliers'
)

sns.scatterplot(
    data=df[(df['is_outlier'] == True) & (df['topology_frequency_group'] == 'rare')],
    x='Df',
    y='Di',
    marker='x',
    color='red',
    s=40,
    zorder=2,
    alpha=1.0,
    edgecolor='r',
    label='Rare topology outliers'
)

sns.scatterplot(
    data=df[(df['is_outlier'] == True) & (df['topology_frequency_group'] == 'common')],
    x='Df',
    y='Di',
    marker='o',
    color='blue', 
    s=40,
    zorder=1,
    alpha=0.8,
    edgecolor='none',
    label='Common topology outliers'
)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='k', markerfacecolor='blue', markersize=14, alpha=1.0, label='Common topology outliers'),
    Line2D([0], [0], marker='x', color='r', markerfacecolor='red', markersize=14, alpha=1.0, label='Rare topology outliers'),
    Line2D([0], [0], marker='o', color='gray', markerfacecolor='gray', markersize=14, alpha=1.0, label='Non-outliers')  # ADDED NON-OUTLIERS
]

ax.legend(
    handles=legend_elements,
    bbox_to_anchor=(1.05, 0.5),
    loc='center left',
    frameon=False,
)
plt.xlabel('$D_f$', labelpad=10)
plt.ylabel('$D_i$',  labelpad=10)
plt.tight_layout(rect=[0, 0, 0.85, 1])  # Reserve 15% space for legend
plt.savefig("manuscript_assets/figure4_outlier_analysis.pdf", bbox_inches='tight')
plt.close()


# Pareto
complexity = {"geometry_only": 1, "topology_only": 2, "enriched_interpretable": 3, "geometry_plus_topology": 5}
pareto_df = raw_mets[raw_mets['model'] == 'hgb'].groupby('family')['r2'].mean().reset_index()
pareto_df['complexity'] = pareto_df['family'].map(complexity)

plt.figure(figsize=(10*1.1,7*1.1))
ax=sns.scatterplot(data=pareto_df, x='complexity', y='r2', s=800,color='orchid')
names=['Enriched','Geo','Geo + Topo',"Topo"]
for i in range(pareto_df.shape[0]):
    if i==3:
        space=0.07
    else:
        space=-0.07
    plt.text(pareto_df.complexity[i]-0.5, pareto_df.r2[i]+space,names[i] )
plt.tight_layout()
ax.set_xlim(-0.5,8)
ax.set_xlabel('Complexity',fontsize=11*3.5)
ax.set_ylabel('$R^2$',fontsize=11*3.5)

plt.savefig("manuscript_assets/figure5_pareto.pdf")
plt.close()

print("\nAll pipelines complete. Data, Metrics, and Visualizations saved successfully.")


Executing Regime & Outlier Analysis...

All pipelines complete. Data, Metrics, and Visualizations saved successfully.
